In [64]:
!pip install sentence-transformers

In [1]:
corpus = "Hi whats your name. I am Soumya. You are watching the SP Insite. "

In [2]:
from nltk.tokenize import sent_tokenize

In [3]:
import nltk
nltk.download('punkt_tab')
documents = sent_tokenize(corpus)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [4]:
documents

['Hi whats your name.', 'I am Soumya.', 'You are watching the SP Insite.']

In [5]:
from nltk.tokenize import word_tokenize

In [6]:
words = word_tokenize(corpus)

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
df = pd.read_csv("movies_metadata.csv")

/tmp/ipykernel_653/1650877637.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("movies_metadata.csv")


In [9]:
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [11]:
df.isnull().sum()

,0
adult,0
belongs_to_collection,40972
budget,0
genres,0
homepage,37684
id,0
imdb_id,17
original_language,11
original_title,0
overview,954


In [12]:
df.duplicated().sum()

np.int64(13)

In [13]:
df = df.drop_duplicates().reset_index(drop=True)

In [14]:
df = df.dropna(subset=['title', 'video', 'vote_average', 'vote_count'])

In [15]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [16]:
df.loc[:, 'overview'] = df['overview'].fillna('No overview found.')
# Ensure the column is string type and aggressively strip all leading/trailing whitespace
df.loc[:, 'overview'] = df['overview'].astype(str).str.strip()

# Use a single regex replacement for 'No Overview', 'No movie overview available.', and empty strings
# The regex ^(...|...|)$ matches the entire string if it's one of the options or an empty string.
problematic_patterns_final = r'^(No Overview|No movie overview available.)$|^$'
df.loc[:, 'overview'] = df['overview'].str.replace(
    problematic_patterns_final,
    'No overview found.',
    regex=True
)

In [17]:
df['belongs_to_collection'] = df['belongs_to_collection'].fillna('None')

In [18]:

import ast


def extract_genres(text):
    try:
        # Convert the string representation of a list into an actual Python list
        genre_list = ast.literal_eval(text)

        # Loop through the list, grab the 'name' value, and join them with commas
        return ", ".join([genre['name'] for genre in genre_list])

    except (ValueError, TypeError):
        # If the data is missing (NaN) or corrupted, return an empty string
        return ""

# Apply the function to your 'genres' column
df['genres'] = df['genres'].apply(extract_genres)


In [19]:
import ast
import pandas as pd

# Generic function to extract 'name' from a string representation of a list of dictionaries
def extract_names_from_list_of_dicts(text):
    if pd.isna(text) or text == '[]' or text == '':
        return ""
    try:
        items = ast.literal_eval(text)
        if isinstance(items, list):
            return ", ".join([item['name'] for item in items if isinstance(item, dict) and 'name' in item])
        return "" # If literal_eval returns something other than a list
    except (ValueError, TypeError, SyntaxError):
        return ""

# Specific function for belongs_to_collection, which is typically a single dictionary or 'None'
def extract_collection_name(text):
    if text == 'None' or pd.isna(text) or text == '':
        return "None"
    try:
        collection_dict = ast.literal_eval(text)
        if isinstance(collection_dict, dict) and 'name' in collection_dict:
            return collection_dict['name']
        return "None" # If literal_eval returns something other than a dict or lacks 'name'
    except (ValueError, TypeError, SyntaxError):
        return "None"

# Apply the generic function to the specified columns
df['spoken_languages'] = df['spoken_languages'].apply(extract_names_from_list_of_dicts)
df['production_countries'] = df['production_countries'].apply(extract_names_from_list_of_dicts)
df['production_companies'] = df['production_companies'].apply(extract_names_from_list_of_dicts)

# Apply the specific function to 'belongs_to_collection'
df['belongs_to_collection'] = df['belongs_to_collection'].apply(extract_collection_name)

In [20]:
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,Toy Story Collection,30000000,"Animation, Comedy, Family",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,English,Released,NaN,Toy Story,False,7.7,5415.0
1,False,None,65000000,"Adventure, Fantasy, Family",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"English, Français",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,Grumpy Old Men Collection,0,"Romance, Comedy",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,English,Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,None,16000000,"Comedy, Drama, Romance",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,English,Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,Father of the Bride Collection,0,Comedy,NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,English,Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45447 entries, 0 to 45452
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45447 non-null  object 
 1   belongs_to_collection  45447 non-null  object 
 2   budget                 45447 non-null  object 
 3   genres                 45447 non-null  object 
 4   homepage               7776 non-null   object 
 5   id                     45447 non-null  object 
 6   imdb_id                45430 non-null  object 
 7   original_language      45436 non-null  object 
 8   original_title         45447 non-null  object 
 9   overview               45447 non-null  object 
 10  popularity             45447 non-null  object 
 11  poster_path            45064 non-null  object 
 12  production_companies   45447 non-null  object 
 13  production_countries   45447 non-null  object 
 14  release_date           45363 non-null  object 
 15  revenue

In [22]:
df['popularity'] = pd.to_numeric(df['popularity'], errors='coerce')

In [23]:
df = df[['adult', 'belongs_to_collection', 'genres',
        'overview', 'title',
        'popularity', 'production_companies',
        'spoken_languages', 'status', 'tagline',
        'vote_average', 'vote_count']].copy()

In [24]:
df['production_companies'] = df['production_companies'].replace('', 'Unknown')

In [25]:
df['tagline'] = df['tagline'].fillna('None')
df['tagline'] = df['tagline'].astype(str)
df['tagline'] = df['tagline'].str.strip()
df['tagline'] = df['tagline'].replace('', 'None')

In [26]:
df['tagline'] = df['tagline'].replace('-', 'None')

In [27]:
df = df[df['status'] != 'Canceled']

In [28]:
df['status'] = df['status'].fillna(df['status'].mode()[0])

In [29]:
df['genres'] = df['genres'].replace("", "Unknown").fillna("Unknown")

In [30]:
df['spoken_languages'] = df['spoken_languages'].replace("", "Unknown").fillna("Unknown")

In [58]:
for col in df.columns:
    if df[col].dtype == 'object':
      print(df[col].value_counts())
      print("-" * 50)

adult
False    45436
True         9
Name: count, dtype: int64
--------------------------------------------------
belongs_to_collection
None                              40956
The Bowery Boys                      29
Totò Collection                      27
James Bond Collection                26
Zatôichi: The Blind Swordsman        26
                                  ...  
Чебурашка и крокодил Гена             1
Tomtar och Trolltyg Collection        1
Hailey Dean Mystery Collection        1
Kocan Kadar Konuş Serisi              1
Ivan Brovkin Collection               1
Name: count, Length: 1696, dtype: int64
--------------------------------------------------
genres
Drama                                           4999
Comedy                                          3621
Documentary                                     2723
Unknown                                         2440
Drama, Romance                                  1300
                                                ... 
Family, A

In [59]:
print(df['adult'].unique())

['False' 'True']


As you can see, the 'adult' column contains string values ('True', 'False') and potentially other non-boolean values (like 'adult'). We need to convert this column to a proper boolean type.

In [60]:
def convert_adult_to_bool(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return value.lower() == 'true'
    return False # Default for any other unexpected values

df['adult'] = df['adult'].apply(convert_adult_to_bool)

print(df['adult'].value_counts())

adult
False    45436
True         9
Name: count, dtype: int64


Now that the `adult` column is properly converted to boolean, let's see the titles where `adult` is `True`.

In [61]:
adult_movies_titles_corrected = df[df['adult'] == True]['title']
display(adult_movies_titles_corrected)

,title
19484,Erotic Nights of the Living Dead
28689,Standoff
31918,Electrical Girl
32097,Diet of Sex
39882,Amateur Porn Star Killer 2
39883,The Band
40553,The Sinful Dwarf
40988,Adulterers
43069,Half -Life


In [32]:
df.head()

,adult,belongs_to_collection,genres,overview,title,popularity,production_companies,spoken_languages,status,tagline,vote_average,vote_count
0,False,Toy Story Collection,"Animation, Comedy, Family","Led by Woody, Andy's toys live happily in his ...",Toy Story,21.946943,Pixar Animation Studios,English,Released,None,7.7,5415.0
1,False,None,"Adventure, Fantasy, Family",When siblings Judy and Peter discover an encha...,Jumanji,17.015539,"TriStar Pictures, Teitler Film, Interscope Com...","English, Français",Released,Roll the dice and unleash the excitement!,6.9,2413.0
2,False,Grumpy Old Men Collection,"Romance, Comedy",A family wedding reignites the ancient feud be...,Grumpier Old Men,11.712900,"Warner Bros., Lancaster Gate",English,Released,Still Yelling. Still Fighting. Still Ready for...,6.5,92.0
3,False,None,"Comedy, Drama, Romance","Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale,3.859495,Twentieth Century Fox Film Corporation,English,Released,Friends are the people who let you be yourself...,6.1,34.0
4,False,Father of the Bride Collection,Comedy,Just when George Banks has recovered from his ...,Father of the Bride Part II,8.387519,"Sandollar Productions, Touchstone Pictures",English,Released,Just When His World Is Back To Normal... He's ...,5.7,173.0


In [33]:
df["tags"] = df['overview'] + df['genres'] + df['tagline']

In [34]:
df['tags'][0]

"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.Animation, Comedy, FamilyNone"

In [35]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import regex as re
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [36]:
def preprocing(text):
  text = str(text).lower()
  text = re.sub(r'[^\w\s]','',text)
  stop = set(stopwords.words('english'))
  text = " ".join([word for word in str(text).split() if word not in stop])
  lemmatizer = WordNetLemmatizer()
  text = " ".join([lemmatizer.lemmatize(word) for word in text.split()])
  return text


In [37]:
df['tags'] = df['tags'].apply(preprocing)

In [38]:
df['tags'][0]

'led woody andys toy live happily room andys birthday brings buzz lightyear onto scene afraid losing place andys heart woody plot buzz circumstance separate buzz woody owner duo eventually learns put aside differencesanimation comedy familynone'

In [39]:
df = df.reset_index(drop=True)

In [40]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

In [41]:
indices

,0
title,
Toy Story,0
Jumanji,1
Grumpier Old Men,2
Waiting to Exhale,3
Father of the Bride Part II,4
...,...
Subdue,45440
Century of Birthing,45441
Betrayal,45442


In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [43]:
tfidf = TfidfVectorizer(stop_words='english',max_features=50000,ngram_range=(1,2))

In [44]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [45]:
tfidf_matrix.shape

(45445, 50000)

In [46]:
print(tfidf.get_feature_names_out())

['00' '000' '007' ... 'он' 'по' 'что']


In [47]:
df.columns

Index(['adult', 'belongs_to_collection', 'genres', 'overview', 'title',
       'popularity', 'production_companies', 'spoken_languages', 'status',
       'tagline', 'vote_average', 'vote_count', 'tags'],
      dtype='object')

In [48]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_recommendations(title, df, tfidf_matrix, indices, top_n=10):
    """
    Recommends movies based on collection membership and TF-IDF cosine similarity.
    """
    # 1. Validate if the movie exists in our dataset
    if title not in indices:
        return f"Sorry, '{title}' was not found in the dataset."

    # 2. Get the index of the movie
    idx = indices[title]

    # Handle edge case where duplicate titles exist by taking the first one
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    # 3. Collection Logic Priority
    movie_collection = df.loc[idx, 'belongs_to_collection']
    collection_recs = []

    if movie_collection != 'None':
        # Grab all other movies in the same collection
        collection_df = df[(df['belongs_to_collection'] == movie_collection) & (df['title'] != title)]
        # Sort them chronologically or by popularity if you wish, but here we just grab their titles
        collection_recs = collection_df['title'].tolist()

    # 4. NLP/TF-IDF Cosine Similarity Logic
    # Compute similarity ONLY for the requested movie to save memory
    # tfidf_matrix[idx] is a 1x50000 sparse matrix
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    # Get the indices of the highest scoring movies
    # We grab a larger pool to account for items we might filter out (like the movie itself or collection duplicates)
    pool_size = top_n + len(collection_recs) + 5
    similar_indices = sim_scores.argsort()[-pool_size:][::-1]

    # 5. Combine and Deduplicate
    nlp_recs = []
    for i in similar_indices:
        rec_title = df.loc[i, 'title']
        # Don't recommend the movie itself, and don't duplicate collection movies
        if i != idx and rec_title not in collection_recs and rec_title != title:
            nlp_recs.append(rec_title)

    # Priority 1: Collection. Priority 2: NLP Similarities
    final_recs = collection_recs + nlp_recs

    # Return exactly the number requested
    return final_recs[:top_n]

In [63]:
# --- Test it out! ---
# Test 1: A movie with a collection
print("Recommendations for 'Toy Story':")
print(get_recommendations('Toy Story', df, tfidf_matrix, indices, top_n=10))

print("\n" + "="*40 + "\n")

# Test 2: A stand-alone movie
print("Recommendations for 'Waiting to Exhale':")
print(get_recommendations('Waiting to Exhale', df, tfidf_matrix, indices, top_n=10))

Recommendations for 'Toy Story':
['Toy Story 2', 'Toy Story 3', 'Small Fry', 'Group Sex', 'Rebel Without a Cause', 'Malice', 'Condorman', 'Hot Splash', 'The Sunchaser', 'Andy Kaufman Plays Carnegie Hall']


Recommendations for 'Waiting to Exhale':
['A Cry in the Night', "Goin' All the Way!", 'Heartbreaker', 'The Scarf', 'Woman in Love', 'Hero', 'Woman of the Lake', "I Don't Buy Kisses Anymore", 'Robin of Locksley', 'En Büyük Şaban']
